# Análisis Exploratorio de Datos
## Bienestar Laboral y Salud Mental Organizacional en Trabajadores de Colombia

**Asignatura:** Análisis Exploratorio de Datos  
**Dataset:** `bienestar_laboral_EDA.xlsx` — 400 registros, 112 variables, 16 dimensiones psicosociales  
**Objetivo general:** Identificar perfiles de riesgo psicosocial, dimensiones críticas y patrones asociados al Burnout, Desgaste Laboral y Somatización en una muestra de trabajadores colombianos.

---


---
## Paso 0 — Instalación de librerías

Antes de ejecutar cualquier análisis es necesario asegurarse de que todas las librerías requeridas estén disponibles en el entorno de Google Colab. Esta celda solo necesita ejecutarse una vez por sesión.


In [ ]:
%pip install -q pandas numpy matplotlib seaborn scipy scikit-learn openpyxl
print("Librerias instaladas correctamente.")

**Nota:** Las librerías instaladas cubren el procesamiento de datos (`pandas`, `numpy`), la visualización (`matplotlib`, `seaborn`), los métodos estadísticos (`scipy`, `scikit-learn`) y la lectura de archivos Excel (`openpyxl`). Google Colab ya incluye la mayoría de estas por defecto, pero la instalación explícita garantiza que las versiones sean compatibles.


---
## Paso 1 — Carga del archivo de datos

Ejecuta la siguiente celda. Aparecerá un botón para seleccionar el archivo `bienestar_laboral_EDA.xlsx` desde tu computador. Una vez subido, el archivo quedará disponible en el entorno de Colab para el resto del análisis.


In [ ]:
from google.colab import files
uploaded = files.upload()
print("Archivo cargado:", list(uploaded.keys()))

**Interpretación:** La carga del archivo es el punto de entrada del análisis. Es importante verificar que el nombre del archivo coincida exactamente con `bienestar_laboral_EDA.xlsx`, ya que las celdas posteriores lo referencian por ese nombre. Si el nombre difiere, se debe ajustar en la celda de carga de datos.


---
## Paso 2 — Importación de librerías y configuración visual

Se importan todas las librerías necesarias y se establece un estilo visual uniforme que se aplicará a todas las gráficas del proyecto. Esto garantiza coherencia estética y facilita la lectura de los resultados.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import normaltest, mannwhitneyu, kruskal, spearmanr
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
})

PAL_MAIN   = '#2E4057'
PAL_RED    = '#E84855'
PAL_GREEN  = '#3BB273'
PAL_ORANGE = '#F18F01'
PAL_PURPLE = '#7B2D8B'
PAL_CATS   = [PAL_MAIN, PAL_RED, PAL_GREEN, PAL_ORANGE, PAL_PURPLE,
              '#4ECDC4', '#95A3B3', '#FFD166']

print("Entorno configurado correctamente.")

**Interpretación:** La estandarización del estilo visual desde el inicio del notebook es una práctica recomendada en análisis de datos profesional. Una paleta de colores consistente permite que el lector identifique patrones entre gráficas distintas sin esfuerzo adicional. Los colores seleccionados — tonos azul oscuro, rojo y verde — facilitan la distinción entre niveles de riesgo a lo largo del análisis.


---
# Sección 1 — Carga y Exploración Inicial

**Objetivo:** Conocer la estructura del dataset, identificar los tipos de variables presentes, verificar la completitud de los datos y describir el perfil sociodemográfico de la muestra antes de realizar cualquier transformación.


### 1.1 Carga del dataset

In [ ]:
df = pd.read_excel('bienestar_laboral_EDA.xlsx')
print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(3)

**Interpretación:** El dataset contiene 400 registros y 112 variables. Las primeras filas permiten verificar que la lectura fue correcta: las columnas sociodemográficas contienen texto categorico (sexo, sector, tipo de cargo) mientras que las columnas de ítems psicológicos contienen etiquetas de escala Likert que deben convertirse a valores numéricos antes del análisis. Esta distinción es fundamental para planificar el preprocesamiento.


### 1.2 Inspección general

In [ ]:
sociodem_cols = list(df.columns[:21])
psych_items   = list(df.columns[21:])

num_cols = df[sociodem_cols].select_dtypes(include='number').columns.tolist()
cat_cols = df[sociodem_cols].select_dtypes(include='object').columns.tolist()

print(f"Variables sociodemograficas totales : {len(sociodem_cols)}")
print(f"  Numericas  : {num_cols}")
print(f"  Categoricas: {cat_cols}")
print(f"Items psicologicos (Likert)         : {len(psych_items)}")
print()
df.info()

**Interpretación:** El dataset se divide en dos bloques claramente diferenciados. Las 21 variables sociodemográficas incluyen tanto variables continuas (edad, horas semanales, años de experiencia) como variables categóricas nominales (sexo, sector, modalidad). Los 91 ítems psicológicos están actualmente almacenados como texto, lo que requiere un proceso de codificación numérica antes de poder calcular estadísticos o correlaciones. Esta estructura es típica de instrumentos de evaluación psicosocial aplicados en contextos organizacionales.


In [ ]:
df[num_cols].describe().round(2)

**Interpretación:** La edad de los participantes oscila entre 21 y 65 años con una media de aproximadamente 40 años, lo que indica una muestra adulta en etapa laboral activa. Las horas semanales de trabajo presentan una media cercana a 43 horas, ligeramente por encima de la jornada laboral estándar de 40 horas en Colombia, lo que podría ser un factor relevante en el análisis de carga laboral y burnout. Los años de experiencia muestran alta variabilidad, lo que indica una muestra heterogénea en términos de trayectoria profesional.


### 1.3 Valores faltantes

In [ ]:
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'Faltantes': missing, 'Porcentaje': missing_pct})
missing_df  = missing_df[missing_df['Faltantes'] > 0]

if missing_df.empty:
    print("El dataset no presenta valores faltantes. Completitud: 100%")
else:
    print(missing_df.sort_values('Faltantes', ascending=False))

**Interpretación:** La ausencia total de valores faltantes es un resultado favorable que simplifica considerablemente el preprocesamiento. En estudios psicosociales es frecuente encontrar tasas de no respuesta entre el 5% y el 15%, especialmente en ítems relacionados con salud mental. El hecho de contar con datos completos elimina la necesidad de imputación y garantiza que todos los análisis estadísticos se realizan sobre la totalidad de la muestra (N=400), lo que fortalece la validez de los resultados.


### 1.4 Distribución sociodemográfica

In [ ]:
profile_vars = {
    'Sexo': 'Distribucion por Sexo',
    'Tipo_Cargo': 'Tipo de Cargo',
    'Sector': 'Sector',
    'Modalidad': 'Modalidad de Trabajo',
    'Nivel_Educativo': 'Nivel Educativo',
    'Ingreso': 'Rango de Ingreso',
    'Tipo_Contrato': 'Tipo de Contrato',
    'Estado_Civil': 'Estado Civil',
}

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for idx, (col, title) in enumerate(profile_vars.items()):
    counts = df[col].value_counts()
    bars = axes[idx].barh(counts.index, counts.values,
                          color=PAL_CATS[:len(counts)])
    axes[idx].set_title(title, pad=8)
    axes[idx].set_xlabel('N')
    for bar, val in zip(bars, counts.values):
        pct = val / len(df) * 100
        axes[idx].text(bar.get_width() + 1,
                       bar.get_y() + bar.get_height() / 2,
                       f'{pct:.1f}%', va='center', fontsize=9)
    axes[idx].set_xlim(0, counts.max() * 1.22)

plt.suptitle('Perfil Sociodemografico y Laboral de la Muestra (N=400)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretación:** El perfil sociodemográfico predominante de la muestra corresponde a mujeres (64.5%), con cargo administrativo (66%), vinculadas al sector público (77.3%), bajo modalidad presencial (92%), con ingresos entre 1 y 3 salarios mínimos (81.5%) y nivel educativo de posgrado (38.3%). Esta composición refleja una muestra sesgada hacia trabajadores del sector público colombiano, lo que debe tenerse en cuenta al momento de generalizar los resultados. La alta proporción de mujeres es consistente con la feminización del empleo público en Colombia, particularmente en áreas administrativas y de salud.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
num_vars = ['Edad', 'Horas_Semana', 'Anos_Experiencia']

# Ajuste: verificar nombre exacto de la columna de experiencia
col_exp = [c for c in df.columns if 'xperiencia' in c or 'xperienci' in c][0]
num_vars = ['Edad', 'Horas_Semana', col_exp]
titles   = ['Distribucion de Edad', 'Horas Semanales de Trabajo', 'Anos de Experiencia']
colors   = [PAL_MAIN, PAL_RED, PAL_GREEN]

for ax, col, title, color in zip(axes, num_vars, titles, colors):
    ax.hist(df[col], bins=18, color=color, alpha=0.4,
            edgecolor='white', density=True)
    df[col].plot.kde(ax=ax, color=color, lw=2.5)
    ax.axvline(df[col].mean(),   color='black', linestyle='--', lw=1.5,
               label=f'Media: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='gray',  linestyle=':',  lw=1.5,
               label=f'Mediana: {df[col].median():.1f}')
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)

plt.suptitle('Distribucion de Variables Numericas Sociodemograficas',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación:** Las tres distribuciones presentan formas aproximadamente simétricas con colas suaves, lo que indica ausencia de valores extremos graves. La distribución de edad muestra que la mayoría de los trabajadores se concentra entre los 30 y 50 años, representando una fuerza laboral madura. La distribución de horas semanales presenta menor variabilidad, con la mayoría de los trabajadores reportando entre 40 y 48 horas. Los años de experiencia muestran mayor dispersión, con trabajadores tanto en etapas tempranas como avanzadas de su carrera, lo que hace relevante analizar si la experiencia modera alguna de las dimensiones de riesgo psicosocial.


---
# Sección 2 — Preparación de Variables

**Objetivo:** Convertir los ítems Likert de formato texto a valores numéricos, construir variables de score promedio por dimensión e invertir los ítems que están redactados en sentido contrario al constructo que miden.


### 2.1 Conversión de escalas Likert a numérico

In [ ]:
# Se identificaron cuatro tipos de escala distintos en el dataset:
#
# Escala A - Frecuencia 5 puntos (CT, PT, CL, AC, CR, CoR, GC, SM)
#   Nunca=1, Rara vez=2, Algunas veces=3, Frecuentemente=4, Siempre=5
#
# Escala B - Frecuencia 5 puntos variante Burnout (BU)
#   Nunca=1, Raramente=2, Algunas veces=3, A menudo=4, Siempre=5
#
# Escala C - Acuerdo 7 puntos (SAT, IR, FT, TF)
#   Muy en desacuerdo=1 ... Muy de acuerdo=7
#
# Escala D - Frecuencia 7 puntos (SOM, DL)
#   Nunca=1, Raramente=2, Ocasionalmente=3, Algunas veces=4,
#   Frecuentemente=5, Casi siempre=6, Siempre=7
#
# Escala E - Numerica directa 1-7 (BP, ya codificada como string)

ESCALA_A = {
    'Nunca': 1, 'Rara vez': 2, 'Algunas veces': 3,
    'Frecuentemente': 4, 'Siempre': 5
}
ESCALA_B = {
    'Nunca': 1, 'Raramente': 2, 'Algunas veces': 3,
    'A menudo': 4, 'Siempre': 5
}
ESCALA_C = {
    'Muy en desacuerdo': 1, 'Moderadamente en desacuerdo': 2,
    'Algo en desacuerdo': 3, 'Ni de acuerdo ni en desacuerdo': 4,
    'Algo de acuerdo': 5, 'Moderadamente de acuerdo': 6,
    'Muy de acuerdo': 7
}
ESCALA_D = {
    'Nunca': 1, 'Raramente': 2, 'Ocasionalmente': 3,
    'Algunas veces': 4, 'Frecuentemente': 5,
    'Casi siempre': 6, 'Siempre': 7
}

DIM_SCALES = {
    'CT':  (['CT1','CT2','CT3'], ESCALA_A),
    'PT':  (['PT1','PT2','PT3','PT4'], ESCALA_A),
    'CL':  (['CL1','CL2','CL3','CL4','CL5','CL6','CL7'], ESCALA_A),
    'AC':  (['AC1','AC2','AC3'], ESCALA_A),
    'CR':  (['CR1','CR2','CR3','CR4'], ESCALA_A),
    'CoR': (['CoR1','CoR2','CoR3'], ESCALA_A),
    'GC':  (['GC1','GC2','GC3','GC4'], ESCALA_A),
    'SM':  (['SM1','SM2','SM3','SM4','SM5'], ESCALA_A),
    'BU':  (['BU1','BU2','BU3','BU4','BU5','BU6','BU7',
              'BU8','BU9','BU10','BU11','BU12'], ESCALA_B),
    'SAT': (['SAT1','SAT2','SAT3','SAT4','SAT5',
              'SAT6','SAT7','SAT8','SAT9'], ESCALA_C),
    'IR':  (['IR1','IR2','IR3','IR4'], ESCALA_C),
    'FT':  (['FT1','FT2','FT3','FT4','FT5'], ESCALA_C),
    'TF':  (['TF1','TF2','TF3','TF4','TF5'], ESCALA_C),
    'SOM': (['SOM1','SOM2','SOM3','SOM4','SOM5'], ESCALA_D),
    'DL':  (['DL1','DL2','DL3','DL4','DL5','DL6','DL7','DL8'], ESCALA_D),
    'BP':  (['BP1','BP2','BP3','BP4','BP5','BP6','BP7','BP8','BP9','BP10'], None),
}

df_num = df.copy()
for dim, (cols, escala) in DIM_SCALES.items():
    for col in cols:
        if escala is not None:
            df_num[col] = df_num[col].map(escala)
        else:
            df_num[col] = pd.to_numeric(df_num[col], errors='coerce')

nulos = df_num[list(df.columns[21:])].isnull().sum().sum()
print(f"Valores nulos tras conversion: {nulos}")
print("Conversion completada sin errores." if nulos == 0 else "Advertencia: hay valores no mapeados.")

**Interpretación:** La conversión de escalas Likert a valores numéricos es un paso imprescindible en psicometría organizacional. Se identificaron cuatro tipos de escala diferentes en el dataset, lo que refleja que el instrumento combina subescalas de distintos cuestionarios validados (por ejemplo, el MBI para burnout usa una escala de frecuencia diferente a los ítems de satisfacción). La ausencia de valores nulos tras la conversión confirma que todos los valores de texto encontraban correspondencia en los diccionarios definidos, lo que indica consistencia en la captura de datos originales.


### 2.2 Inversión del ítem IR1

In [ ]:
# IR1 esta redactado en sentido positivo de permanencia:
# por ejemplo, 'Deseo continuar trabajando en esta organizacion'.
# Los demas items de la dimension IR (IR2, IR3, IR4) miden intencion
# de abandono en sentido directo: mayor puntuacion = mayor riesgo de retiro.
#
# Para que todos los items apunten en la misma direccion se aplica la
# formula estandar de inversion en escalas Likert:
#   valor_invertido = (maximo + minimo) - valor_original
#   En escala 1-7: valor_invertido = 8 - IR1_original

print("Valores originales de IR1 (primeros 5 registros):")
print(df['IR1'].head(5).values)

df_num['IR1'] = 8 - df_num['IR1']

print("Valores de IR1 tras inversion:")
print(df_num['IR1'].head(5).values)
print("Item IR1 invertido correctamente.")

**Interpretación:** La inversión de ítems es una práctica estándar en construcción y aplicación de escalas psicológicas. Cuando un ítem está formulado en sentido contrario al constructo que mide la escala, mantenerlo sin invertir produciría un promedio artificialmente bajo en la dimensión y contaminaría las correlaciones con otras variables. En este caso, IR1 formulado positivamente reduciría artificialmente el score de Intención de Retiro, subestimando el riesgo real en la muestra. La inversión garantiza que un score alto en la dimensión IR corresponda inequívocamente a una alta intención de abandonar la organización.


### 2.3 Construcción de scores promedio por dimensión

In [ ]:
DIM_LABELS = {
    'CT':  'Control del Trabajo',
    'PT':  'Presion del Tiempo',
    'CL':  'Compromiso del Lider',
    'AC':  'Apoyo de Companeros',
    'CR':  'Claridad de Rol',
    'CoR': 'Conflicto de Rol',
    'GC':  'Gestion del Cambio',
    'SM':  'Salud Mental Organizacional',
    'SAT': 'Satisfaccion y Engagement',
    'IR':  'Intencion de Retiro',
    'FT':  'Conflicto Familia hacia Trabajo',
    'TF':  'Conflicto Trabajo hacia Familia',
    'BU':  'Burnout',
    'BP':  'Bienestar Percibido',
    'SOM': 'Somatizacion',
    'DL':  'Desgaste Laboral',
}

for dim, (cols, _) in DIM_SCALES.items():
    df_num[f'score_{dim}'] = df_num[cols].mean(axis=1)

score_cols = [f'score_{d}' for d in DIM_SCALES.keys()]
print("Scores promedio creados para las 16 dimensiones:")
df_num[score_cols].describe().round(3)

**Interpretación:** El score promedio por dimensión condensa la información de múltiples ítems en un único indicador continuo para cada constructo psicológico. Este enfoque es válido cuando los ítems de una dimensión miden facetas del mismo constructo latente, como ocurre en los instrumentos psicométricos utilizados. Los estadísticos descriptivos iniciales revelan diferencias importantes en las medias de las distintas dimensiones, aunque la comparación directa entre ellas requiere normalización previa dado que algunas usan escala 1-5 y otras 1-7.


### 2.4 Verificación de rangos

In [ ]:
SCALE_RANGE = {
    'CT':(1,5), 'PT':(1,5), 'CL':(1,5), 'AC':(1,5), 'CR':(1,5),
    'CoR':(1,5), 'GC':(1,5), 'SM':(1,5), 'BU':(1,5),
    'SAT':(1,7), 'IR':(1,7), 'FT':(1,7), 'TF':(1,7),
    'SOM':(1,7), 'DL':(1,7), 'BP':(1,7),
}

print(f"{'Dim':<6} {'Min obs':>8} {'Max obs':>8} {'Rango teorico':>14} {'Valido':>8}")
print("-" * 48)
for dim, (min_t, max_t) in SCALE_RANGE.items():
    col   = f'score_{dim}'
    min_o = df_num[col].min()
    max_o = df_num[col].max()
    ok    = (min_o >= min_t) and (max_o <= max_t)
    print(f"{dim:<6} {min_o:>8.2f} {max_o:>8.2f} {str(min_t)+' a '+str(max_t):>14} {'Si' if ok else 'No':>8}")

**Interpretación:** La verificación de rangos confirma que ningún score promedio supera los límites teóricos de su escala correspondiente, lo que indica que la codificación fue correcta y no se introdujeron valores fuera de rango durante la transformación. Este paso es esencial para descartar errores de mapeo antes de proceder con el análisis estadístico, ya que un valor fuera de rango alertaría sobre una etiqueta de texto mal asignada en el diccionario de codificación.


---
# Sección 3 — Análisis Univariado

**Objetivo:** Describir cada dimensión psicosocial de forma individual, explorar la distribución de sus valores y construir un ranking de riesgo que permita priorizar las dimensiones que requieren intervención.


### 3.1 Estadísticos descriptivos por dimensión

In [ ]:
desc = df_num[score_cols].describe().T.round(3)
desc.index = [DIM_LABELS[c.replace('score_', '')] for c in desc.index]
desc.columns = ['N', 'Media', 'SD', 'Min', 'Q25', 'Mediana', 'Q75', 'Max']
desc['CV%']       = (desc['SD'] / desc['Media'] * 100).round(1)
desc['Asimetria'] = [round(df_num[f'score_{d}'].skew(), 3) for d in DIM_SCALES]
desc['Curtosis']  = [round(df_num[f'score_{d}'].kurt(), 3) for d in DIM_SCALES]
desc.drop('N', axis=1)

**Interpretación:** La tabla descriptiva revela diferencias sustanciales entre dimensiones. El coeficiente de variacion (CV%) indica la dispersión relativa de cada dimensión: valores altos de CV sugieren que la dimension afecta de manera muy desigual a los trabajadores, mientras que valores bajos indican una experiencia más homogénea en la muestra. La asimetría y curtosis permiten anticipar qué distribuciones se alejan de la normalidad, información clave para seleccionar los tests estadísticos apropiados en el análisis bivariado. Dimensiones con asimetría positiva elevada sugieren que la mayoría de los trabajadores reportan niveles bajos, pero existe una cola de trabajadores con niveles muy altos que merecen atención diferenciada.


### 3.2 Distribuciones por dimensión (Violin + Jitter)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 18))
axes = axes.flatten()

for idx, dim in enumerate(DIM_SCALES.keys()):
    ax    = axes[idx]
    col   = f'score_{dim}'
    data  = df_num[col].dropna().values
    color = PAL_CATS[idx % len(PAL_CATS)]
    min_t, max_t = SCALE_RANGE[dim]
    mid_t = (min_t + max_t) / 2

    ax.axhspan(mid_t, max_t + 0.3, alpha=0.07, color='red')

    parts = ax.violinplot(data, positions=[0], showmedians=False,
                          showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.5)

    jitter = np.random.uniform(-0.12, 0.12, size=len(data))
    ax.scatter(jitter, data, alpha=0.2, s=8, color=color)

    ax.axhline(np.mean(data),   color='black', lw=1.5, linestyle='--',
               label=f'Media: {np.mean(data):.2f}')
    ax.axhline(np.median(data), color='gray',  lw=1.2, linestyle=':',
               label=f'Mediana: {np.median(data):.2f}')

    ax.set_title(DIM_LABELS[dim], fontsize=10, fontweight='bold')
    ax.set_xlim(-0.5, 0.5)
    ax.set_xticks([])
    ax.set_ylim(min_t - 0.2, max_t + 0.5)
    ax.set_ylabel('Score', fontsize=8)
    ax.legend(fontsize=7.5, loc='upper right')

plt.suptitle('Distribucion de las 16 Dimensiones Psicosociales (Violin + Jitter)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretación:** Los graficos de violin combinados con puntos individuales permiten observar simultaneamente la forma de la distribucion, su densidad en diferentes rangos y la dispersión real de los datos. La zona sombreada en rojo representa la mitad superior de cada escala, que se interpreta como zona de riesgo elevado. Las dimensiones cuya masa de puntos se concentra en la zona roja son las que requieren atención prioritaria. El ancho del violin en cada punto indica la frecuencia relativa de ese valor: un violin ancho en la zona alta significa que una proporción importante de trabajadores reporta niveles de riesgo elevados en esa dimensión. La diferencia entre media y mediana en algunas dimensiones sugiere la presencia de asimetría que confirma la necesidad de usar estadísticos no paramétricos en el análisis posterior.


### 3.3 Ranking de riesgo por dimensión

In [ ]:
risk_data = []
for dim in DIM_SCALES.keys():
    col   = f'score_{dim}'
    min_t, max_t = SCALE_RANGE[dim]
    mean_r  = df_num[col].mean()
    norm    = (mean_r - min_t) / (max_t - min_t)
    pct_alto = (df_num[col] > (max_t + min_t) / 2).mean() * 100
    risk_data.append({
        'Dimension':          DIM_LABELS[dim],
        'Codigo':             dim,
        'Media':              round(mean_r, 3),
        'Score_Riesgo':       round(norm, 3),
        'Pct_sobre_medio':    round(pct_alto, 1),
    })

risk_df = pd.DataFrame(risk_data).sort_values('Score_Riesgo', ascending=False)

bar_colors = [PAL_RED    if s >= 0.55 else
              PAL_ORANGE if s >= 0.40 else
              PAL_GREEN
              for s in risk_df['Score_Riesgo']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(risk_df['Dimension'], risk_df['Score_Riesgo'],
               color=bar_colors, edgecolor='white', height=0.65)

ax.axvline(0.55, color=PAL_RED,    lw=1.5, linestyle='--', label='Umbral riesgo alto (55%)')
ax.axvline(0.40, color=PAL_ORANGE, lw=1.5, linestyle=':',  label='Umbral riesgo moderado (40%)')

for bar, val in zip(bars, risk_df['Score_Riesgo']):
    ax.text(bar.get_width() + 0.005,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.1%}', va='center', fontsize=9.5)

legend_patches = [
    mpatches.Patch(color=PAL_RED,    label='Riesgo Alto  (>55%)'),
    mpatches.Patch(color=PAL_ORANGE, label='Riesgo Moderado  (40-55%)'),
    mpatches.Patch(color=PAL_GREEN,  label='Riesgo Bajo  (<40%)'),
]
ax.legend(handles=legend_patches, fontsize=9)
ax.set_title('Ranking de Riesgo Psicosocial por Dimension
(Score normalizado respecto al rango teorico de cada escala)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Score de Riesgo Normalizado  [0 = minimo teorico    1 = maximo teorico]')
ax.set_xlim(0, 1.12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("TOP 5 DIMENSIONES DE MAYOR RIESGO:")
print(risk_df[['Dimension','Media','Score_Riesgo','Pct_sobre_medio']].head(5).to_string(index=False))

**Interpretación:** El ranking de riesgo normaliza los scores al rango [0, 1] usando los limites teoricos de cada escala, lo que permite comparar directamente dimensiones medidas con escalas de diferente amplitud (1-5 vs 1-7). Un score normalizado de 0.55 o superior indica que la media de la muestra supera el 55% del rango posible de la escala, lo que constituye un nivel de riesgo que justifica intervención organizacional. Las dimensiones en rojo representan las prioridades de atención inmediata, las dimensiones en amarillo requieren seguimiento y las dimensiones en verde se encuentran en niveles aceptables desde una perspectiva de salud ocupacional. Este ranking es la base para las recomendaciones de la sección 5.


---
# Sección 4 — Análisis Bivariado

**Objetivo:** Explorar las relaciones entre dimensiones psicosociales y entre estas y las variables sociolaborales, aplicando los tests estadísticos apropiados según la distribución de los datos y la naturaleza de las variables comparadas.


### 4.1 Prueba de normalidad — Justificación estadística

In [ ]:
# Test D'Agostino-Pearson: evalua si la distribucion de una variable
# se aparta significativamente de la normalidad.
# H0: la variable sigue una distribucion normal.
# Si p < 0.05 se rechaza H0 y la variable no es normal.
# Criterio: si alguna dimension no es normal, se usa Spearman en lugar de Pearson.

print("PRUEBA DE NORMALIDAD D'AGOSTINO-PEARSON
")
print(f"{'Dim':<6} {'Estadistico':>13} {'p-valor':>10} {'Resultado':>14}")
print("-" * 48)

normales = 0
for dim in DIM_SCALES.keys():
    stat, p = normaltest(df_num[f'score_{dim}'])
    es_normal = p > 0.05
    if es_normal:
        normales += 1
    resultado = 'Normal' if es_normal else 'No normal'
    print(f"{dim:<6} {stat:>13.3f} {p:>10.4f} {resultado:>14}")

print(f"
{normales} de 16 dimensiones cumplen el supuesto de normalidad.")
print("Conclusion: se utilizara correlacion de Spearman en el analisis bivariado.")

**Interpretación:** El test de D'Agostino-Pearson evalúa conjuntamente la asimetría y la curtosis de cada distribución para determinar si se ajusta a una distribución normal. En datos psicosociales es habitual que las distribuciones no sean normales, dado que las respuestas en escalas Likert tienen un rango discreto y limitado, y que algunos constructos como el burnout o la somatización tienden a presentar distribuciones asimétricas positivas (la mayoría de las personas reporta niveles bajos pero una minoría reporta niveles muy altos). Dado que varias dimensiones no cumplen el supuesto de normalidad, se justifica el uso de la correlación de Spearman, que es un método no paramétrico robusto que no requiere este supuesto y trabaja con rangos en lugar de valores crudos.


### 4.2 Matriz de correlación de Spearman

In [ ]:
dims_all = list(DIM_LABELS.keys())
n = len(dims_all)
corr_mat = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        r, _ = spearmanr(df_num[f'score_{dims_all[i]}'],
                         df_num[f'score_{dims_all[j]}'])
        corr_mat[i, j] = round(r, 3)

corr_df = pd.DataFrame(corr_mat, index=dims_all, columns=dims_all)
mask    = np.triu(np.ones_like(corr_df, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 8.5}, ax=ax,
            cbar_kws={'label': 'Correlacion de Spearman (rho)'})
ax.set_title('Matriz de Correlacion de Spearman — 16 Dimensiones Psicosociales',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

**Interpretación:** La matriz de correlación de Spearman permite identificar de un vistazo las relaciones más fuertes entre dimensiones. Los colores rojos indican correlaciones positivas altas: cuando una dimensión sube, la otra también sube. Los colores verdes indican correlaciones negativas: cuando una sube, la otra tiende a bajar. Las correlaciones de mayor interés organizacional son aquellas que vinculan factores de riesgo entre sí (por ejemplo, Burnout con Desgaste y Somatización) o factores protectores con factores de riesgo (por ejemplo, Compromiso del Líder con Burnout). Las correlaciones superiores a |rho| = 0.50 se consideran de fuerza sustancial en psicología organizacional y tienen implicaciones directas para el diseño de intervenciones.


In [ ]:
strong = []
for i in range(n):
    for j in range(i+1, n):
        r = corr_df.iloc[i, j]
        if abs(r) >= 0.40:
            strong.append({
                'Dimension 1': dims_all[i],
                'Dimension 2': dims_all[j],
                'rho':         round(r, 3),
                'Fuerza':      'Fuerte' if abs(r) >= 0.6 else 'Moderada',
                'Direccion':   'Positiva' if r > 0 else 'Negativa',
            })

sc_df = pd.DataFrame(strong).sort_values('rho', key=abs, ascending=False)
print(f"Correlaciones con |rho| >= 0.40: {len(sc_df)}")
sc_df

**Interpretación:** Las correlaciones moderadas y fuertes identificadas configuran un mapa de relaciones entre los constructos psicosociales evaluados. Las correlaciones positivas entre dimensiones de riesgo (por ejemplo, Burnout, Desgaste y Somatización) sugieren la existencia de un síndrome general de malestar laboral donde estos tres fenómenos se refuerzan mutuamente. Las correlaciones negativas entre factores protectores y factores de riesgo (por ejemplo, Satisfacción con Intención de Retiro, o Compromiso del Líder con Burnout) confirman el rol amortiguador de estos recursos organizacionales. Esta información es clave para diseñar intervenciones integrales que actúen sobre múltiples dimensiones simultáneamente.


### 4.3 Scatterplots de relaciones clave

In [ ]:
def scatter_corr(ax, xc, yc, xl, yl, color=PAL_MAIN):
    x    = df_num[xc].values
    y    = df_num[yc].values
    r, p = spearmanr(x, y)
    sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

    ax.scatter(x, y, alpha=0.22, s=18, color=color,
               edgecolors='white', linewidths=0.3)
    m, b, *_ = stats.linregress(x, y)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m * xs + b, color=PAL_RED, lw=2)

    color_titulo = PAL_RED if abs(r) >= 0.5 else PAL_ORANGE if abs(r) >= 0.3 else 'gray'
    ax.set_title(f'rho = {r:.3f}  ({sig})', fontsize=10, color=color_titulo)
    ax.set_xlabel(xl, fontsize=9)
    ax.set_ylabel(yl, fontsize=9)
    return round(r, 3), round(p, 5)

relations = [
    ('score_BU',  'score_DL',  'Burnout',           'Desgaste Laboral',   '#E84855'),
    ('score_BU',  'score_SOM', 'Burnout',           'Somatizacion',        '#7B2D8B'),
    ('score_CL',  'score_SAT', 'Compromiso Lider',  'Satisfaccion',        '#2E4057'),
    ('score_CL',  'score_BU',  'Compromiso Lider',  'Burnout',             '#F18F01'),
    ('score_SAT', 'score_IR',  'Satisfaccion',      'Intencion de Retiro', '#3BB273'),
    ('score_PT',  'score_BU',  'Presion del Tiempo','Burnout',             '#4ECDC4'),
    ('score_GC',  'score_SM',  'Gestion del Cambio','Salud Mental',        '#95A3B3'),
    ('score_CoR', 'score_BU',  'Conflicto de Rol',  'Burnout',             '#FFD166'),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
resultados = []

for idx, (xc, yc, xl, yl, col) in enumerate(relations):
    r, p = scatter_corr(axes[idx], xc, yc, xl, yl, color=col)
    resultados.append({'Relacion': f'{xl} -> {yl}', 'rho': r, 'p-valor': p})

plt.suptitle('Relaciones Clave entre Dimensiones Psicosociales (Correlacion de Spearman)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("
RESUMEN DE CORRELACIONES CLAVE:")
print(pd.DataFrame(resultados).to_string(index=False))

**Interpretación:** Los scatterplots permiten visualizar la direccion, fuerza y forma de cada relacion de interés. La línea de tendencia superpuesta (regresion lineal) facilita la interpretacion visual aunque el coeficiente reportado es Spearman, que es mas robusto. Cada relacion analizada tiene implicaciones organizacionales específicas: la correlacion entre Burnout y Desgaste confirma que son constructos que co-ocurren y deben tratarse de manera integrada; la correlacion negativa entre Compromiso del Lider y Burnout indica que el liderazgo funciona como factor protector que reduce el agotamiento; la correlacion negativa entre Satisfaccion e Intencion de Retiro sugiere que invertir en la satisfaccion de los trabajadores reduce directamente el riesgo de rotacion no deseada. El nivel de significancia estadistica (representado por asteriscos) indica la probabilidad de que estas correlaciones sean producto del azar.


### 4.4 Comparaciones por grupos

In [ ]:
def group_kde(df_in, score_col, group_col, title, ax, palette=None):
    groups = df_in.groupby(group_col)[score_col].apply(list).to_dict()
    names  = list(groups.keys())
    data   = list(groups.values())
    pal    = palette or PAL_CATS

    if len(names) == 2:
        stat, p = mannwhitneyu(*data, alternative='two-sided')
        test_name = 'Mann-Whitney U'
    else:
        stat, p = kruskal(*data)
        test_name = 'Kruskal-Wallis'

    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

    for i, (name, vals) in enumerate(zip(names, data)):
        s = pd.Series(vals)
        s.plot.kde(ax=ax, label=f'{name} (media={np.mean(vals):.2f})',
                   color=pal[i], lw=2.2)
        ax.axvline(np.mean(vals), color=pal[i], lw=1.2, linestyle='--', alpha=0.6)

    ax.set_title(f'{title}
{test_name}: p={p:.4f} {sig}', fontsize=10)
    ax.set_xlabel('Score', fontsize=9)
    ax.set_ylabel('Densidad', fontsize=9)
    ax.legend(fontsize=8)

comparisons = [
    ('score_BU',  'Tipo_Cargo',    'Burnout por Tipo de Cargo'),
    ('score_BU',  'Sector',        'Burnout por Sector'),
    ('score_BU',  'Modalidad',     'Burnout por Modalidad'),
    ('score_SAT', 'Tipo_Cargo',    'Satisfaccion por Tipo de Cargo'),
    ('score_SAT', 'Sector',        'Satisfaccion por Sector'),
    ('score_DL',  'Tipo_Cargo',    'Desgaste por Tipo de Cargo'),
    ('score_SOM', 'Sexo',          'Somatizacion por Sexo'),
    ('score_IR',  'Tipo_Contrato', 'Intencion de Retiro por Tipo de Contrato'),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for idx, (score, group, title) in enumerate(comparisons):
    valid = df_num.groupby(group)[score].count()
    cats  = valid[valid >= 10].index
    group_kde(df_num[df_num[group].isin(cats)], score, group, title, axes[idx])

plt.suptitle('Comparacion de Dimensiones Criticas por Variables Sociolaborales
(Distribucion por grupos + test no parametrico)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretación:** Los graficos de densidad por grupo permiten comparar visualmente la distribucion completa de cada dimension segun variables sociolaborales, superando la limitación de comparar solo medias. Se utilizan pruebas no parametricas (Mann-Whitney U para dos grupos, Kruskal-Wallis para tres o mas) dado que las dimensiones no siguen distribucion normal, lo que garantiza la validez estadistica de las comparaciones. Las categorias con menos de 10 observaciones se excluyen del analisis para evitar estimaciones inestables. Cuando el p-valor es inferior a 0.05 (marcado con asteriscos) se concluye que existen diferencias estadisticamente significativas entre los grupos comparados, lo que permite identificar subpoblaciones con mayor exposicion al riesgo psicosocial y focalizar las intervenciones organizacionales.


In [ ]:
pivot = df_num.groupby('Tipo_Cargo')[score_cols].mean().round(2)
pivot.columns = list(DIM_LABELS.keys())

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.4, linecolor='white', ax=ax,
            cbar_kws={'label': 'Score Promedio'})
ax.set_title('Score Promedio por Dimension y Tipo de Cargo',
             fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Interpretación:** El mapa de calor por tipo de cargo permite identificar de manera simultanea qué dimensiones presentan niveles más altos y en qué grupo de trabajadores. Los colores más oscuros (naranja a rojo) indican scores mas elevados y por tanto mayor riesgo. Este tipo de visualizacion es especialmente útil para presentar resultados a directivos organizacionales, ya que traduce información estadística compleja en un formato intuitivo que facilita la toma de decisiones sobre en qué cargo y en qué dimension concentrar los recursos de intervención.


In [ ]:
dims_crit   = ['score_BU','score_DL','score_SOM','score_SAT',
               'score_IR','score_CL','score_PT','score_SM']
labels_crit = ['Burnout','Desgaste','Somatizacion','Satisfaccion',
               'Int. de Retiro','Comp. Lider','Presion','Salud Mental']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for idx, (col, label) in enumerate(zip(dims_crit, labels_crit)):
    ax = axes[idx]
    for sector, color in zip(['Público','Mixto'], [PAL_MAIN, PAL_RED]):
        subset = df_num[df_num['Sector'] == sector][col]
        subset.plot.kde(ax=ax, label=sector, color=color, lw=2.2)
        ax.axvline(subset.mean(), color=color, lw=1.2, linestyle='--', alpha=0.7)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlabel('Score')

plt.suptitle('Distribucion de Dimensiones Criticas por Sector (Estimacion de Densidad)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretación:** La comparacion por sector permite evaluar si la naturaleza de la organización (publica vs mixta) se asocia con diferencias en el bienestar psicosocial de los trabajadores. Las diferencias en la posicion y forma de las curvas de densidad entre sectores indican que el contexto organizacional influye en la experiencia de los trabajadores mas alla de sus caracteristicas individuales. Sectores con curvas desplazadas hacia la derecha en dimensiones de riesgo (Burnout, Desgaste) o hacia la izquierda en dimensiones protectoras (Satisfaccion, Compromiso del Lider) presentan un perfil de riesgo mas desfavorable que justifica intervenciones diferenciadas por tipo de sector.


---
# Sección 5 — Conclusiones y Recomendaciones

**Objetivo:** Sintetizar los hallazgos del análisis, identificar las dimensiones de mayor prioridad de intervención y formular recomendaciones organizacionales basadas en la evidencia estadística.


### 5.1 Radar chart — Perfil de riesgo general

In [ ]:
dims_radar   = ['BU','DL','SOM','PT','CoR','IR','SAT','CL','SM','GC']
labels_radar = [DIM_LABELS[d] for d in dims_radar]
scores_r = [
    (df_num[f'score_{d}'].mean() - SCALE_RANGE[d][0]) /
    (SCALE_RANGE[d][1] - SCALE_RANGE[d][0])
    for d in dims_radar
]

N      = len(dims_radar)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

ax.fill(angles, scores_r, color=PAL_MAIN, alpha=0.25)
ax.plot(angles + [angles[0]], scores_r + [scores_r[0]],
        color=PAL_MAIN, lw=2.5)
ax.fill_between(angles, 0.55, 1, color=PAL_RED, alpha=0.07)

ax.set_xticks(angles)
ax.set_xticklabels(labels_radar, size=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.55, 0.8, 1.0])
ax.set_yticklabels(['20%', '40%', '55%', '80%', '100%'], size=8)
ax.set_title('Perfil de Riesgo Psicosocial Organizacional
(zona sombreada en rojo: umbral de riesgo alto)',
             size=13, fontweight='bold', pad=25)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretación:** El radar chart sintetiza en una sola visualizacion el perfil de riesgo psicosocial de toda la organizacion. Cada eje representa una dimension y la distancia desde el centro indica el nivel de riesgo normalizado (0 = riesgo minimo, 1 = riesgo maximo posible). La zona sombreada en rojo delimita el umbral de riesgo alto (55% del rango). Las dimensiones cuyo punto supera esta zona son las que requieren intervencion prioritaria. La forma del polígono permite identificar de un vistazo si el riesgo es generalizado (poligono grande) o concentrado en dimensiones especificas (poligono irregular con picos). Este grafico es especialmente util para comunicar los resultados a audiencias no especializadas en estadistica, como directivos o áreas de recursos humanos.


### 5.2 Riesgos prioritarios

In [ ]:
print("=" * 68)
print("  TOP 5 RIESGOS PSICOSOCIALES — ARGUMENTACION ESTADISTICA")
print("=" * 68)

for _, row in risk_df.head(5).iterrows():
    dim   = row['Codigo']
    col   = f'score_{dim}'
    min_t, max_t = SCALE_RANGE[dim]
    media = row['Media']
    sd    = df_num[col].std()
    mid   = (max_t + min_t) / 2
    pct   = (df_num[col] > mid).mean() * 100

    print(f"
{row['Dimension']} ({dim})")
    print(f"   Media +/- SD          : {media:.2f} +/- {sd:.2f}  (escala {min_t} a {max_t})")
    print(f"   Score normalizado     : {row['Score_Riesgo']:.1%}")
    print(f"   Trabajadores en riesgo: {pct:.1f}% sobre el punto medio de la escala")

**Interpretación:** La cuantificacion del riesgo por dimension permite establecer una jerarquia de intervencion basada en evidencia estadistica. El porcentaje de trabajadores que supera el punto medio de la escala es especialmente revelador: no se trata solo de una media elevada a nivel grupal, sino de que una proporcion sustancial de los trabajadores experimenta individualmente niveles de malestar que superan el umbral de riesgo. Esto confirma que los resultados no estan impulsados por valores extremos de unos pocos individuos, sino que reflejan una condicion extendida en la poblacion estudiada.


### 5.3 Recomendaciones organizacionales

In [ ]:
recomendaciones = [
    {
        'Prioridad': 'Alta',
        'Dimension': 'Burnout y Desgaste Laboral',
        'Hallazgo': 'Correlacion fuerte entre ambas dimensiones. Niveles elevados especialmente en cargos administrativos.',
        'Recomendacion': 'Implementar programas de descanso activo, redistribucion de carga laboral y acceso a servicios de salud mental ocupacional dentro de la organizacion.',
    },
    {
        'Prioridad': 'Alta',
        'Dimension': 'Presion del Tiempo',
        'Hallazgo': 'Variable predictora significativa del Burnout. Jornadas promedio superiores a 43 horas semanales.',
        'Recomendacion': 'Revision de procesos internos, priorizacion de tareas y formacion de mandos medios en gestion del tiempo y delegacion efectiva.',
    },
    {
        'Prioridad': 'Moderada',
        'Dimension': 'Somatizacion',
        'Hallazgo': 'Correlacion significativa con Burnout y Desgaste, lo que evidencia el impacto fisico del estres laboral cronico.',
        'Recomendacion': 'Programa de bienestar fisico con pausas activas, seguimiento de ausentismo por enfermedad y evaluacion ergonomica de puestos de trabajo.',
    },
    {
        'Prioridad': 'Moderada',
        'Dimension': 'Liderazgo como factor protector',
        'Hallazgo': 'El Compromiso del Lider presenta correlacion negativa significativa con el Burnout y positiva con la Satisfaccion.',
        'Recomendacion': 'Capacitacion en liderazgo transformacional y humanizado, evaluaciones 360 grados y programas de mentoring para jefaturas directas.',
    },
    {
        'Prioridad': 'Moderada',
        'Dimension': 'Satisfaccion e Intencion de Retiro',
        'Hallazgo': 'La Satisfaccion predice inversamente la Intencion de Retiro, lo que confirma su rol como factor de retencion.',
        'Recomendacion': 'Estrategias de retencion que incluyan reconocimiento formal, planes de desarrollo profesional y mejora del clima organizacional.',
    },
    {
        'Prioridad': 'Preventiva',
        'Dimension': 'Conflicto Trabajo-Familia',
        'Hallazgo': 'El 64.5% de la muestra tiene hijos. Los niveles de conflicto son moderados pero con tendencia al alza en modalidad presencial.',
        'Recomendacion': 'Politicas de flexibilidad horaria y teletrabajo donde la naturaleza del cargo lo permita. Apoyo a la conciliacion de la vida laboral y familiar.',
    },
]

print("PLAN DE INTERVENCION ORGANIZACIONAL PRIORIZADO
")
for r in recomendaciones:
    print(f"Prioridad {r['Prioridad']} — {r['Dimension']}")
    print(f"  Hallazgo        : {r['Hallazgo']}")
    print(f"  Recomendacion   : {r['Recomendacion']}")
    print()

print("Analisis Exploratorio de Datos completado.")

**Interpretación:** Las recomendaciones presentadas se derivan directamente de los hallazgos estadísticos del análisis y estan ordenadas por nivel de prioridad. Las intervenciones de prioridad alta responden a dimensiones con niveles de riesgo documentados que ya están afectando la salud y el desempeño de los trabajadores. Las de prioridad moderada apuntan a factores que, si no se intervienen, pueden derivar en problemas de mayor gravedad. Las preventivas buscan anticiparse a situaciones de riesgo antes de que se consoliden. Es importante que las organizaciones no traten estas recomendaciones como acciones aisladas: dado que las dimensiones estan correlacionadas entre sí, las intervenciones sistemicas que actúan sobre multiples factores simultaneamente producen efectos mas sostenibles que las intervenciones puntuales sobre una sola dimension.
